# Phase 4A: Real 15-Case Blind Comparative Evaluation Notebook

**Target Model**: `google/gemma-2-2b-it` (Requires Hugging Face token `HF_TOKEN` for gated access)  
**Investigator Model**: `gemini-3.8-flash` via Google GenAI SDK (`google-genai`)  

### Evaluation Objective
Executes a rigorous 15-case comparative benchmark (`cases/evaluation_cases.json`) across 3 failure families (5 cases each):
1. **Negation / Instruction-Binding Failures**
2. **Factual Entity-Substitution Failures**
3. **Output-Format / Constraint Failures**

Compares:
- **Agent #1 (Transcript-Only Baseline)**: Receives prompt failure transcript and public expected behavior.
- **Agent #2 (Causal Investigator)**: Receives failure transcript, Agent #1 hypotheses, and controlled residual-stream activation intervention sandbox (`run_target`, `capture`, `patch`, `ablate`).

Both agents freeze their blind predictions **BEFORE** the hidden variant is revealed or executed.

In [ ]:
# 1. Install Dependencies
!pip install -r requirements.txt

In [ ]:
# 2. Run Dry-Run Mock Validation (Orchestration Pipeline Check)
!python scripts/phase4a_evaluation.py --mock-gemma --mock-gemini

In [ ]:
# 3. Set API Credentials for Real Execution (Google Colab Secrets or Environment)
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('  [SUCCESS] GEMINI_API_KEY and HF_TOKEN loaded from Colab secrets.')
except Exception as e:
    print('  [NOTICE] Colab secrets not available:', e)
    # Set manually if needed:
    # os.environ['GEMINI_API_KEY'] = 'AIzaSy...'
    # os.environ['HF_TOKEN'] = 'hf_...'

In [ ]:
# 4. Execute Full 15-Case Benchmark Evaluation (Real Gemma 2 2B IT + Real Gemini 3.8 Flash)
!python scripts/phase4a_evaluation.py

In [ ]:
# 5. Display Evaluation Summary Metrics Report
import json
with open('results/phase4a_summary.json', 'r') as f:
    summary = json.load(f)

print('=' * 60)
print('  PHASE 4A AGGREGATE SUMMARY')
print('=' * 60)
print(f"Target Model      : {summary.get('target_model_name')}")
print(f"Investigator Model: {summary.get('investigator_model_name')}")
print(f"Execution Mode    : {summary.get('execution_mode')}")
print(f"Total Cases       : {summary.get('total_cases')}")
print(f"Agent #1 Accuracy : {summary.get('agent1_accuracy')*100:.1f}%")
print(f"Agent #2 Accuracy : {summary.get('agent2_accuracy')*100:.1f}%")
print(f"Accuracy Delta    : {summary.get('accuracy_delta')*100:+.1f}%")
print('Contingency Breakdown:', json.dumps(summary.get('contingency_breakdown'), indent=2))
print('Accuracy by Family   :', json.dumps(summary.get('accuracy_by_family'), indent=2))
print('Brier Scores         :', json.dumps(summary.get('brier_score'), indent=2))